Environment Setup & Library Imports

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings

warnings.filterwarnings('ignore')
print("[SYSTEM] Environment ready. Himalayan Tech & Gaming analysis engine online.")

Data Lake Generation (Raw Data with Noise)

In [ ]:
os.makedirs('nepal_datalake', exist_ok=True)
np.random.seed(88) # Fixed seed for reproducibility
num_rows = 7500

products = ["4K-TV", "VR-Headset", "Smart-Hub", "Gaming-Console", "Solar-Battery", "Drone-HD"]
cities = ["Kathmandu-DurbarMarg", "Pokhara-Lakeside", "Lalitpur-Patan", "Butwal-Chowk", "Biratnagar-Main"]
addresses = {
    "Kathmandu-DurbarMarg": "King's Way 101", "Pokhara-Lakeside": "Baidam Street 6",
    "Lalitpur-Patan": "Pulchowk Rd", "Butwal-Chowk": "Traffic Chowk E", "Biratnagar-Main": "Hospital Rd"
}

# Source A: Messy JSON (Web Transactions)
json_data = []
for i in range(num_rows):
    loc = np.random.choice(cities + ["UNKNOWN", None])
    price = np.random.uniform(300, 4500)
    if np.random.rand() < 0.05: price = "REFUND_ERROR" # Noise: String in numeric column
    
    json_data.append({
        "txn_ref": f"WEB-{i:05d}",
        "log_meta": {"timestamp": (pd.to_datetime('2023-01-01') + pd.Timedelta(days=np.random.randint(0, 360))).strftime('%Y-%m-%d'), "zone": loc},
        "basket": [{"item": np.random.choice(products), "quantity": np.random.randint(1, 4), "unit_price": price}]
    })
with open('nepal_datalake/web_raw.json', 'w') as f:
    json.dump(json_data, f)

# Source B: CSV Logs (Physical Store POS)
pos_rows = []
for i in range(num_rows):
    city = np.random.choice(cities)
    pos_rows.append({
        "ID": f"POS-{i:05d}",
        "Date": (pd.to_datetime('2023-01-01') + pd.Timedelta(days=np.random.randint(0, 360))).strftime('%Y-%m-%d'),
        "City": city,
        "Street": addresses.get(city),
        "SKU": np.random.choice(products),
        "Units": np.random.randint(1, 10),
        "Cost_Per_Unit": np.random.uniform(-100, 5000) # Noise: Negative prices
    })
pd.DataFrame(pos_rows).to_csv('nepal_datalake/pos_raw.csv', index=False)
print("[SYSTEM] Data Lake created with Himalayan retail logs.")

Attribute Inspection & EDA (Exploratory Data Analysis)

In [ ]:
# Extract and Flatten JSON
with open('nepal_datalake/web_raw.json', 'r') as f:
    df_json_raw = pd.json_normalize(json.load(f), record_path=['basket'], meta=[['log_meta', 'timestamp'], ['log_meta', 'zone'], 'txn_ref'])

# Load CSV
df_csv_raw = pd.read_csv('nepal_datalake/pos_raw.csv')

print("--- SOURCE A (JSON) INSPECTION ---")
print(df_json_raw.info())
print(df_json_raw.head(10)) # Head() for report

print("\n--- SOURCE B (CSV) INSPECTION ---")
print(df_csv_raw.dtypes) # Identify dimensions vs measures

The ETL Pipeline (Transformation & Cleaning)

In [ ]:
# 1. Standardization
df_json_raw.columns = ['Product', 'Qty', 'Price', 'Date', 'Location', 'ID']
df_csv_raw.columns = ['ID', 'Date', 'Location', 'Address', 'Product', 'Qty', 'Price']

# 2. Integration
master_df = pd.concat([df_json_raw, df_csv_raw], ignore_index=True)

# 3. Data Quality Gate
master_df['Price'] = pd.to_numeric(master_df['Price'], errors='coerce')
master_df['Qty'] = pd.to_numeric(master_df['Qty'], errors='coerce').fillna(1)

# Drop noise and duplicates
master_df = master_df.dropna(subset=['Price', 'Location'])
master_df = master_df[(master_df['Price'] > 0) & (master_df['Location'] != "UNKNOWN")]

# 4. Feature Engineering (Creating the Cube structure)
master_df['Date'] = pd.to_datetime(master_df['Date'])
master_df['Quarter'] = 'Q' + master_df['Date'].dt.quarter.astype(str)
master_df['Year'] = master_df['Date'].dt.year
master_df['Revenue'] = master_df['Price'] * master_df['Qty']
master_df['Address'] = master_df['Address'].fillna(master_df['Location'].map(addresses))

print(f"[ETL] Cleaning complete. Total trustworthy records: {len(master_df)}")

OLAP Operations (Slice, Dice, Roll-Up, Drill-Down)


In [ ]:
# 1. SLICE: Filtering for Q1 Sales across all products and locations
slice_q1 = master_df[master_df['Quarter'] == 'Q1']

# 2. DICE: Focusing on '4K-TV' in 'Kathmandu' during 'Q1'
dice_ktm_tv_q1 = master_df[
    (master_df['Product'] == '4K-TV') & 
    (master_df['Location'].str.contains('Kathmandu')) & 
    (master_df['Quarter'] == 'Q1')
]

# 3. DRILL-DOWN: Moving from City to Store Address
drill_down = master_df.groupby(['Location', 'Address'])['Revenue'].sum().reset_index()

# 4. ROLL-UP: Moving from Daily to Yearly totals
roll_up = master_df.groupby('Year')['Revenue'].sum().reset_index()

print("--- DICE RESULT: 4K-TV in Kathmandu (Q1) ---")
print(dice_ktm_tv_q1.head()) # Screenshot this for report section

Visual Evidence (Data Cube & Heatmap)

In [ ]:
# 3D Data Cube Visualization
fig_cube = px.scatter_3d(master_df, x='Location', y='Product', z='Quarter', 
                         size='Revenue', color='Revenue', 
                         title="Nepal Sales Data Cube: Multi-Dimensional View")
fig_cube.show() # Screenshot for visual evidence

# Pivot Heatmap
pivot_tab = master_df.pivot_table(index='Product', columns='Location', values='Revenue', aggfunc='sum')
plt.figure(figsize=(10,6))
sns.heatmap(pivot_tab, annot=False, cmap='magma')
plt.title("Himalayan Tech: Revenue Performance Heatmap")
plt.show() # Screenshot for visual evidence

Addressing the Curse of Dimensionality (PCA)

In [ ]:
# Prepare for dimensionality reduction
df_pca = master_df.copy()
df_pca['P_Enc'] = df_pca['Product'].astype('category').cat.codes
df_pca['L_Enc'] = df_pca['Location'].astype('category').cat.codes

# Scaling features
features = ['Qty', 'Price', 'P_Enc', 'L_Enc']
x = StandardScaler().fit_transform(df_pca[features])

# PCA
pca = PCA(n_components=2)
components = pca.fit_transform(x)

plt.figure(figsize=(8,6))
plt.scatter(components[:,0], components[:,1], c=df_pca['Revenue'], cmap='viridis', alpha=0.5)
plt.title("PCA: Combatting the Curse of Dimensionality")
plt.colorbar(label='Revenue')
plt.show() # Screenshot for your report